# Шаг 1. Готовая модель YOLO находит объекты на фотографии

В этом ноутбуке мы возьмём уже обученную нейросеть **YOLO** (You Only Look Once) и попросим её найти объекты на картинке.
Обучать ничего не нужно: модель уже умеет распознавать 80 классов объектов из набора данных **COCO** —
людей, машины, автобусы, собак, чашки, телефоны и т.д.

Что мы сделаем:
1. Загрузим модель.
2. Выберем картинку (пример из интернета или своё фото).
3. Запустим распознавание.
4. Посмотрим, что нашла модель: класс объекта, уверенность и координаты рамки.
5. Нарисуем рамки на картинке и сохраним результат.

In [ ]:
# Настройка для Google Colab. На своём компьютере эта ячейка ничего не делает.
# В Colab нужна папка machine_vision на Google Диске (в ней session01–session05 и dataset).
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive:
    drive.mount("/content/drive")
    COLAB_ROOT = "/content/drive/MyDrive/machine_vision"
    os.chdir(f"{COLAB_ROOT}/session01")  # здесь лежит common.py и сохраняются результаты
    sys.path.insert(0, os.getcwd())
    import subprocess

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

## 1. Импорт библиотек

- `ultralytics` — библиотека, в которой живёт YOLO. Класс `YOLO` загружает модель и запускает её.
- `labels_ru` — наш маленький модуль из этой же папки: функция `ru()` переводит английские названия классов на русский
  (`"bus"` → `"автобус"`).

> Если `ultralytics` не установлена, выполните `pip install -r requirements.txt`.

In [ ]:
from ultralytics import YOLO

from labels_ru import ru

## 2. Выбор модели

Модели YOLO бывают разных размеров: `n` (nano), `s` (small), `m` (medium), `l` (large), `x` (extra large).
Чем больше модель, тем она точнее, но медленнее.

Мы берём `yolov8n.pt` — **самую маленькую и быструю**. Она отлично работает даже на ноутбуке без видеокарты.
Если файла модели нет рядом, он **скачается автоматически** при первом запуске.

> Можно попробовать `yolo11n.pt` — это более новая версия, код при этом не меняется.

In [ ]:
# Готовая модель, обученная на 80 классах набора COCO.
MODEL = "yolov8n.pt"

model = YOLO(MODEL)

## 3. Выбор картинки

`source` — это то, на чём модель будет искать объекты. Это может быть:
- ссылка на картинку в интернете;
- путь к файлу на компьютере, например `"мое_фото.jpg"`;
- даже путь к видео или номер веб-камеры (но об этом — в следующем шаге).

По умолчанию возьмём классическую тестовую картинку с автобусом. Чтобы проверить своё фото,
положите его в эту папку и замените строку ниже.

In [ ]:
source = "https://ultralytics.com/images/bus.jpg"
# source = "мое_фото.jpg"   # раскомментируйте, чтобы использовать своё фото

## 4. Запуск распознавания

Вызов `model(source)` прогоняет картинку через нейросеть. Модель возвращает **список результатов** —
по одному на каждую картинку. У нас картинка одна, поэтому берём первый элемент `[0]`.

Параметр `conf` — **порог уверенности** (от 0 до 1). Модель сообщает только о тех объектах,
в которых уверена хотя бы на 25%. Попробуйте поменять порог:
- `conf=0.1` — найдётся больше объектов, но появятся ошибки;
- `conf=0.7` — останутся только самые надёжные находки.

In [ ]:
result = model(source, conf=0.25)[0]  # conf — минимальная уверенность, от 0 до 1

## 5. Что нашла модель

Найденные объекты лежат в `result.boxes`. Для каждой рамки (`box`) есть:

| Поле | Что означает |
|------|--------------|
| `box.cls` | номер класса (например, 0 — человек, 5 — автобус) |
| `box.conf` | уверенность модели, число от 0 до 1 |
| `box.xyxy` | координаты рамки: левый верхний угол `(x1, y1)` и правый нижний `(x2, y2)` в пикселях |

Номер класса превращаем в название через словарь `result.names`, а затем переводим на русский функцией `ru()`.

> Обратите внимание: начало координат `(0, 0)` — в **левом верхнем** углу картинки, ось `y` направлена вниз.

In [ ]:
print(f"Найдено объектов: {len(result.boxes)}")
for box in result.boxes:
    name = result.names[int(box.cls)]               # номер класса -> английское название
    confidence = float(box.conf)                    # уверенность, 0..1
    x1, y1, x2, y2 = (int(v) for v in box.xyxy[0])  # углы рамки в пикселях
    print(f"  {ru(name):<16} уверенность {confidence:.0%}   рамка ({x1}, {y1}) – ({x2}, {y2})")

## 6. Смотрим на картинку с рамками

`result.plot()` рисует рамки и подписи прямо на картинке и возвращает её как массив пикселей.
OpenCV (на котором работает рисование) хранит цвета в порядке **BGR**, а обычные программы — в **RGB**,
поэтому переворачиваем порядок каналов: `[..., ::-1]`.

In [ ]:
from PIL import Image

Image.fromarray(result.plot()[..., ::-1])

## 7. Сохраняем результат

`result.save()` записывает картинку с рамками в файл — её можно открыть любым просмотрщиком изображений.

In [ ]:
result.save("result.jpg")
print("Картинка с рамками сохранена в result.jpg")

## Попробуйте сами

1. Замените `source` на своё фото — что модель находит, а что пропускает?
2. Поиграйте с порогом `conf`: как меняется количество найденных объектов?
3. Замените модель на `yolov8s.pt` или `yolo11n.pt` и сравните результаты и скорость.
4. Найдите объект, которого **нет** среди 80 классов COCO (например, кактус). Что скажет модель?